# Notebook 11 — Range-Bearing Landmark Measurements

This notebook explains the 2D **range-bearing measurement model** used in EKF-SLAM.


## Learning objectives

1. Compute expected range-bearing measurements from robot pose + landmark position.
2. Understand `atan2` and why angle wrapping is mandatory.
3. Inject Gaussian noise in polar coordinates and interpret the geometry in Cartesian.
4. Explain why bearing noise creates a **banana-shaped** cloud in Cartesian space.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from notebooks._notebook_utils import set_seed, wrap_angle

SEED = set_seed(11)
np.set_printoptions(precision=4, suppress=True)
print(f"Seed: {SEED}")


## 1) Geometry of range and bearing

Robot pose is $x_r = [x, y, \theta]^T$, landmark is $m=[m_x, m_y]^T$.

$$
\Delta x = m_x - x,\quad \Delta y = m_y - y,\quad q = \Delta x^2 + \Delta y^2.
$$

$$
h(x_r,m)=\begin{bmatrix} r \\ \phi \end{bmatrix}
=\begin{bmatrix}
\sqrt{q} \\
\mathrm{wrap}(\operatorname{atan2}(\Delta y,\Delta x)-\theta)
\end{bmatrix}.
$$


In [ ]:
def expected_measurement(robot_pose: np.ndarray, landmark_xy: np.ndarray) -> np.ndarray:
    dx = landmark_xy[0] - robot_pose[0]
    dy = landmark_xy[1] - robot_pose[1]
    rng = np.hypot(dx, dy)
    bearing = wrap_angle(np.arctan2(dy, dx) - robot_pose[2])
    return np.array([rng, bearing], dtype=float)

robot = np.array([2.0, 1.0, np.deg2rad(30.0)])
landmark = np.array([5.0, 4.0])
z_hat = expected_measurement(robot, landmark)
print("robot [x,y,theta(rad)]:", robot)
print("landmark [mx,my]:", landmark)
print("expected z=[range(m), bearing(rad)]:", z_hat)
print("bearing(deg):", np.rad2deg(z_hat[1]))


### Interactive mini-lab: change pose + landmark

Edit `robot_user` and `landmark_user` below, then rerun the cell.


In [ ]:
robot_user = np.array([0.0, 0.0, np.deg2rad(170.0)])
landmark_user = np.array([-2.0, 0.4])

z_user = expected_measurement(robot_user, landmark_user)
print("robot:", robot_user)
print("landmark:", landmark_user)
print("expected z:", z_user)
print("bearing(deg):", np.rad2deg(z_user[1]))


## 2) `atan2` and angle wrapping

`atan2(y, x)` returns an angle in $(-\pi,\pi]$. Subtracting heading can push it outside this interval.
Wrap back into $[-\pi,\pi)$:

$$
\mathrm{wrap}(\alpha)= (\alpha+\pi)\bmod 2\pi - \pi.
$$


In [ ]:
angles = np.deg2rad(np.array([-540, -190, -179, 0, 179, 190, 540]))
wrapped = wrap_angle(angles)

for a, w in zip(angles, wrapped):
    print(f"raw={np.rad2deg(a):7.1f} deg  -> wrapped={np.rad2deg(w):7.1f} deg")


In [ ]:
pred_bearing = np.deg2rad(179.0)
meas_bearing = np.deg2rad(-179.0)

residual_bad = meas_bearing - pred_bearing
residual_good = wrap_angle(residual_bad)

print("Unwrapped residual (deg):", np.rad2deg(residual_bad))
print("Wrapped residual   (deg):", np.rad2deg(residual_good))


## 3) Noise in polar coordinates

$$
z = h(x) + v,\quad v\sim\mathcal{N}(0,R),\quad
R = \begin{bmatrix}\sigma_r^2 & 0 \\ 0 & \sigma_\phi^2\end{bmatrix}.
$$


In [ ]:
def polar_to_world(robot_pose, r, b):
    x, y, th = robot_pose
    global_angle = th + b
    return np.column_stack([x + r * np.cos(global_angle), y + r * np.sin(global_angle)])

robot = np.array([0.0, 0.0, np.deg2rad(10.0)])
landmark = np.array([8.0, 0.5])
z_true = expected_measurement(robot, landmark)

sigma_r = 0.2
sigma_b = np.deg2rad(4.0)
R = np.diag([sigma_r**2, sigma_b**2])

N = 3000
noise = np.random.multivariate_normal(mean=np.zeros(2), cov=R, size=N)
z_noisy = z_true + noise
z_noisy[:, 1] = wrap_angle(z_noisy[:, 1])
pts = polar_to_world(robot, z_noisy[:, 0], z_noisy[:, 1])

plt.figure(figsize=(7, 5))
plt.scatter(pts[:, 0], pts[:, 1], s=6, alpha=0.25, label="Noisy samples (Cartesian)")
plt.scatter([landmark[0]], [landmark[1]], c="red", s=80, marker="x", label="True landmark")
plt.scatter([robot[0]], [robot[1]], c="k", s=40, label="Robot")
plt.axis("equal")
plt.grid(True, alpha=0.3)
plt.title("Range-bearing noise sampled in polar, viewed in Cartesian")
plt.legend()
plt.show()


### Numeric check: sample covariance in polar


In [ ]:
sample_cov = np.cov((z_noisy - z_true).T)
print("Target R:\n", R)
print("Sample covariance:\n", sample_cov)
print("Abs error:\n", np.abs(sample_cov - R))


## Exercise — Why banana-shaped uncertainty appears

1. Keep range noise small (e.g., $\sigma_r=0.05$ m), increase bearing noise (e.g., 8–15 deg).
2. Keep bearing noise small, increase range noise.
3. Compare point clouds.

**Reason:** bearing noise rotates the ray around the robot, so lateral spread grows with distance.


In [ ]:
# Optional solution
robot = np.array([0.0, 0.0, 0.0])
landmark = np.array([10.0, 0.0])
z_true = expected_measurement(robot, landmark)
N = 4000

def sample_cloud(sig_r, sig_b_deg):
    R = np.diag([sig_r**2, np.deg2rad(sig_b_deg)**2])
    noise = np.random.multivariate_normal(np.zeros(2), R, size=N)
    z = z_true + noise
    z[:, 1] = wrap_angle(z[:, 1])
    return polar_to_world(robot, z[:, 0], z[:, 1])

cloud_bearing = sample_cloud(sig_r=0.05, sig_b_deg=12.0)
cloud_range = sample_cloud(sig_r=0.8, sig_b_deg=1.0)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(cloud_bearing[:, 0], cloud_bearing[:, 1], s=4, alpha=0.2)
ax[0].scatter([10], [0], c='red', marker='x', s=80)
ax[0].set_title("High bearing noise -> banana-like arc")
ax[0].axis('equal'); ax[0].grid(alpha=0.3)

ax[1].scatter(cloud_range[:, 0], cloud_range[:, 1], s=4, alpha=0.2)
ax[1].scatter([10], [0], c='red', marker='x', s=80)
ax[1].set_title("High range noise -> radial spread")
ax[1].axis('equal'); ax[1].grid(alpha=0.3)
plt.show()


## Recap

- Range-bearing prediction uses geometry plus angle wrapping.
- `atan2` and wrapping are essential for stable residuals.
- Gaussian polar noise can look curved (banana-like) in Cartesian space.
